# Speech Emotion Analysis System - Documentation

## How the System Works (Step by Step)

```
┌─────────────────────────────────────────────────────────────────┐
│                    STEP 1: USER UPLOADS AUDIO                   │
│                                                                 │
│  • User records their voice or uploads an audio file            │
│  • Supported formats: WAV, MP3, M4A, FLAC, OGG, WebM            │
│  • Allowed length: 1 second to 20 minutes                       │
│  • File size limit: 100 MB                                      │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│            STEP 2: PREPARE THE AUDIO FILE                       │
│                                                                 │
│  What happens behind the scenes:                                │
│  • Load the audio file into memory                              │
│  • Convert to standard format (mono - single channel)           │
│  • Resample to 16,000 Hz (industry standard for speech)         │
│  • Normalize volume (make it consistent)                        │
│  • Remove complete silence sections                             │
│                                                                 │
│  Why this matters: The emotion detection model needs audio      │
│  in a specific format to work properly.                         │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│         STEP 3: BREAK AUDIO INTO MANAGEABLE PIECES              │
│                                                                 │
│  If audio is LONGER than 30 seconds:                            │
│  • Split into 30-second chunks                                  │
│  • Add 5-second overlap between chunks                          │
│  • This keeps emotional context consistent                      │
│                                                                 │
│  Example: 60-second audio becomes:                              │
│    Chunk 1: 0-30 seconds                                        │
│    Chunk 2: 25-55 seconds                                       │
│    Chunk 3: 50-60 seconds                                       │
│                                                                 │
│  If audio is 30 seconds or SHORTER:                             │
│  • Process as one complete piece                                │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│       STEP 4: DETECT EMOTIONS (Wav2Vec2 AI Model)               │
│                                                                 │
│  The AI model analyzes each chunk:                              │
│  • Listens to the audio patterns (pitch, speed, tone, etc.)     │
│  • Calculates probability of 8 emotions                         │
│  • Produces emotion scores for each chunk                       │
│                                                                 │
│  The 8 Emotions Detected:                                       │
│  ✓ Angry      (frustrated, upset)                               │
│  ✓ Calm       (peaceful, relaxed)                               │
│  ✓ Disgust    (repulsed, offended)                              │
│  ✓ Fearful    (anxious, scared)                                 │
│  ✓ Happy      (joyful, pleased)                                 │
│  ✓ Neutral    (no emotion, flat)                                │
│  ✓ Sad        (depressed, unhappy)                              │
│  ✓ Surprised  (shocked, astonished)                             │
│                                                                 │
│  Output per chunk (example):                                    │
│    angry: 8%, calm: 15%, disgust: 3%, fearful: 12%              │
│    happy: 42%, neutral: 12%, sad: 5%, surprised: 3%             │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│     STEP 5: COMBINE RESULTS INTO OVERALL EMOTION                │
│                                                                 │
│  Why combine? If we analyzed 5 chunks, we get 5 different       │
│  emotion detections. We need to combine them smartly.           │
│                                                                 │
│  Smart Combination Method:                                      │
│  • Longer chunks get more weight (more representative)          │
│  • Higher confidence emotions get more weight                   │
│  • Formula: Weight = chunk_length × (0.5 + 0.5 × confidence)    │
│                                                                 │
│  Result: Single overall emotion + confidence score              │
│    Final Emotion: HAPPY (42% overall)                           │
│    Confidence: High (0.85 out of 1.0)                           │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│           STEP 6: ASSESS AUDIO QUALITY ⭐ HOW IT WORKS          │
│                                                                 │
│  Audio quality affects emotion detection accuracy. Poor         │
│  quality audio = less reliable results.                         │
│                                                                 │
│  THREE QUALITY CHECKS:                                          │
│                                                                 │
│  1️⃣  ENERGY LEVEL (Volume Consistency)                          │
│  ├─ What: Measure how loud the audio is                         │
│  ├─ Why: Consistent volume = clearer voice signals              │
│  ├─ How: Calculate mean energy across audio                     │
│  ├─ Formula: energy_score = min(1.0, energy × 1000)             │
│  └─ Result: 0.0 (very quiet) to 1.0 (very loud)                 │
│                                                                 │
│  2️⃣  SPECTRAL CENTROID (Clarity & Brightness)                   │
│  ├─ What: Measure clarity of speech (high frequencies)          │
│  ├─ Why: Speech should have balanced frequencies                │
│  ├─ How: Analyze frequency distribution                         │
│  ├─ Formula: centroid_score = min(1.0, centroid / 4000)         │
│  └─ Result: 0.0 (very muffled) to 1.0 (crystal clear)           │
│                                                                 │
│  3️⃣  NOISE ESTIMATION (Background Noise)                        │
│  ├─ What: Detect unwanted background sounds                     │
│  ├─ Why: Noise interferes with emotion detection                │
│  ├─ How: Compare quiet parts to loud parts                      │
│  └─ Result: Noise level estimate                                │
│                                                                 │
│  FINAL QUALITY SCORE:                                           │
│  • Combines energy (60% weight) + clarity (40% weight)          │
│  • Quality = (energy_score × 0.6) + (centroid_score × 0.4)      │
│  • Range: 0.1 (poor) to 1.0 (excellent)                         │
│  • Threshold for acceptance: ≥ 0.6 (60%)                        │
│                                                                 │
│  ❌ Low Quality Triggers:                                       │
│     - Too much background noise (coffee shop, traffic)          │
│     - Very quiet recording (can't hear well)                    │
│     - Robotic or distorted audio                                │
│     - Disconnected or choppy recording                          │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│      STEP 7: GENERATE CLINICAL INSIGHTS                         │
│                                                                 │
│  Based on emotions detected, calculate:                         │
│                                                                 │
│  • Emotional Valence (positive vs negative):                    │
│    Range: -1 (very negative) to +1 (very positive)              │
│    Formula: (positive_emotions - negative_emotions) /           │
│             (positive_emotions + negative_emotions)             │
│                                                                 │
│  • Emotional Arousal (energetic vs calm):                       │
│    Range: 0 (very calm) to 1 (very energetic)                   │
│    Formula: high_energy_emotions / (high + low energy)          │
│                                                                 │
│  • Emotional Stability (confidence level):                      │
│    Range: 0 (very uncertain) to 1 (very certain)                │
│    How dominant emotion is compared to others                   │
│                                                                 │
│  • Clinical Risk Indicators:                                    │
│    ✓ Depression risk (high sadness)                             │
│    ✓ Anxiety risk (high fear, rapid speech)                     │
│    ✓ Social engagement level (emotional variety)                │
│    ✓ Cognitive health markers (speech clarity, fluency)         │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│     STEP 8: ADVANCED AI ANALYSIS (GPT-4) - Optional             │
│                                                                 │
│  If API key available:                                          │
│  • Deep analysis of vocal patterns                              │
│  • Speech coherence evaluation                                  │
│  • Detailed clinical risk assessment                            │
│  • Personalized care recommendations                            │
│                                                                 │
│  If API not available:                                          │
│  • System still works with basic analysis above                 │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│      STEP 9: DECIDE IF EXPERT REVIEW NEEDED                     │
│                                                                 │
│  Automatic Review Trigger if ANY of these:                      │
│  ✓ Low emotion confidence (< 40%)                               │
│  ✓ Poor audio quality (< 40%)                                   │
│  ✓ Very high negative emotions (> 70% sad/scared/angry)         │
│  ✓ High clinical priority emotions detected (≥ 60%)             │
│  ✓ Audio very short (< 2 seconds) or too long (> 20 min)        │
│  ✓ High-risk indicators detected (depression, anxiety)          │
│  ✓ AI suggests human review is needed                           │
│                                                                 │
│  If flagged: Human expert (Speech-Language Pathologist)         │
│             will manually review the analysis                   │
└────────────────────────┬────────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────────┐
│         STEP 10: SAVE RESULTS & SHOW TO USER                    │
│                                                                 │
│  • Save complete session data as JSON file                      │
│  • Store in database for future reference                       │
│  • Create review request if needed                              │
│  • Generate audit log (for compliance/tracking)                 │
│  • Send results back to user/healthcare provider                │
│  • Show detected emotions, confidence, quality score            │
└─────────────────────────────────────────────────────────────────┘
```

---

## 🧪 Audio Quality Assessment - Technical Details

This is one of the most important parts because **garbage in = garbage out**. If the audio quality is bad, the emotion detection will be unreliable.

### What Gets Measured

#### 1. **Energy Level** (How Loud/Clear the Voice Is)
```
Purpose: Ensure audio volume is adequate for analysis
Process:
  1. Load a sample of audio (first 10 seconds)
  2. Calculate average power/energy across samples
  3. Energy = average(audio_sample²)
  4. Convert to score: min(1.0, energy × 1000)

What it detects:
  ✓ Whisper or very quiet recordings (low score)
  ✓ Normal speaking volume (mid-high score)
  ✓ Shouting or very loud (high score)

Example:
  • Whisper: 0.1 (too quiet)
  • Normal talk: 0.7 (good)
  • Shouting: 1.0 (clear but extreme)
```

#### 2. **Spectral Centroid** (Voice Clarity & Richness)
```
Purpose: Ensure audio has good frequency distribution
What it measures: How "bright" or "clear" the speech sounds

Process:
  1. Analyze frequency content of audio
  2. Spectral centroid = where 50% of sound energy is below
  3. Typical human speech: 1000-4000 Hz
  4. Score: min(1.0, centroid / 4000)

What it detects:
  ✓ Muffled/over-compressed audio (low frequency only)
  ✓ Natural human speech (balanced frequencies)
  ✓ Robotic/artificial audio (unusual patterns)

Example:
  • Muffled recording: centroid=1500 → score=0.38
  • Clear speech: centroid=3000 → score=0.75
  • Crystal clear: centroid=4000+ → score=1.0
```

#### 3. **Background Noise** (Implicit in Energy/Clarity)
```
The system detects noise by comparing:
  • Quiet moments vs loud moments
  • Consistent patterns (noise) vs variable patterns (speech)
  • Unexpected low-frequency rumbles (background hum)

What it detects:
  ✓ Traffic noise
  ✓ Fan/AC humming
  ✓ Background chatter
  ✓ Wind noise
  ✓ Radio/TV in background
```

### How Quality Score Is Calculated

```python
# The actual formula used in the code:
energy_score = min(1.0, energy * 1000)  # 0.0 to 1.0
centroid_score = min(1.0, centroid / 4000)  # 0.0 to 1.0

# Combine them (energy more important than clarity)
quality_score = (energy_score * 0.6) + (centroid_score * 0.4)

# Ensure it's always in valid range
quality_score = max(0.1, min(1.0, quality_score))

# Typical results:
# ✓ Good quality audio: 0.7 - 1.0 → Analysis very reliable
# ⚠️ Moderate quality: 0.4 - 0.6 → Results less reliable
# ❌ Poor quality: 0.0 - 0.4 → Triggers human review
```

### Real-World Examples

| Scenario | Quality Score | Recommendation |
|----------|---------------|-----------------|
| Clean indoor recording with good mic | 0.85-1.0 | ✓ Excellent - Proceed |
| Phone call quality | 0.65-0.75 | ✓ Good - Acceptable |
| Recorded in quiet room | 0.70-0.85 | ✓ Good - Acceptable |
| Coffee shop/ambient noise | 0.30-0.50 | ⚠️ Poor - Flag for review |
| Whisper/very quiet talk | 0.20-0.40 | ❌ Very Poor - Reject |
| Robotic/distorted audio | 0.25-0.45 | ❌ Poor - Flag for review |
| Old/compressed recording | 0.40-0.60 | ⚠️ Borderline - Review |

---

## 📦 System Components Explained

### 1. **Wav2Vec2 Predictor** (`wav2vec2_predictor.py`)
**What it is:** The AI model that detects emotions from voice

**What it does:**
- Takes audio file, preprocesses it
- Splits long audio into 30-second chunks
- Analyzes each chunk separately
- Combines results intelligently
- Returns: dominant emotion + confidence + all 8 emotions

**Inside the code:**
- Uses `Wav2Vec2Processor` to prepare audio
- Uses `Wav2Vec2ForSequenceClassification` model (from HuggingFace)
- Runs inference with PyTorch
- Outputs probability distribution (softmax)

### 2. **Speech Analyzer** (`speech_analyzer.py`)
**What it is:** The orchestrator that puts everything together

**What it does:**
- Validates audio files
- Calls Wav2Vec2Predictor for emotion detection
- Assesses audio quality
- Generates clinical insights
- Decides if human review is needed

**Key functions:**
- `analyze()` - Main analysis entry point
- `_assess_audio_quality()` - Runs quality checks
- `get_emotion_insights()` - Generates clinical insights
- `_should_require_review()` - Checks review conditions

### 3. **Speech Analysis Service** (`speech_analysis_service.py`)
**What it is:** The service layer that handles sessions and database

**What it does:**
- Manages user sessions
- Coordinates between components
- Saves results to database
- Handles human review system integration
- Creates audit logs for compliance

**Key functions:**
- `start_audio_session()` - Initialize session
- `analyze_audio_file()` - Full analysis pipeline
- `analyze_audio_stream()` - Handle streaming audio
- `_finalize_audio_session()` - Save all results
- `get_service_statistics()` - System health stats

### 4. **Agentic Reasoner** (`agentic_reasoner.py`)
**What it is:** Advanced AI analysis using GPT-4 (optional)

**What it does:**
- If API key available, does deeper analysis
- Analyzes emotional patterns
- Assesses speech coherence
- Generates clinical risk scores
- Provides personalized recommendations

### 5. **Model Manager** (`model_manager.py`)
**What it is:** Manages AI models and versions

**What it does:**
- Tracks model versions
- Stores performance metrics
- Handles model updates
- Manages storage

### 6. **Continuous Learning** (`continuous_learning.py`)
**What it is:** System for improving the model over time

**What it does:**
- Collects expert feedback
- Retrains model weekly (if enough feedback)
- Tracks improvements
- Ensures model stays accurate

---

## 🧠 How Emotions Are Classified

### The 8 Emotions & Their Meanings

| Emotion | Meaning | Clinical Concern | Example Voice Pattern |
|---------|---------|------------------|----------------------|
| **Angry** | Frustrated, upset | Potential aggression | Tense, sharp, fast |
| **Calm** | Peaceful, relaxed | Positive | Smooth, slow, steady |
| **Disgust** | Repulsed, offended | Moderate concern | Scrunched, tense |
| **Fearful** | Anxious, scared | High concern | Shaky, high-pitched |
| **Happy** | Joyful, pleased | Positive | Bright, energetic |
| **Neutral** | No emotion, flat | Could indicate depression | Monotone, flat |
| **Sad** | Unhappy, depressed | High concern | Slow, low-energy |
| **Surprised** | Shocked, astonished | Low concern | Quick pitch change |

### Grouping for Clinical Use

**Positive Emotions** (Good sign):
- Happy, Calm, Surprised

**Negative Emotions** (Concerning):
- Sad, Angry, Fearful, Disgust

**High Clinical Priority** (Requires attention):
- Sad (depression indicator)
- Fearful (anxiety indicator)
- Disgust (emotional distress)

---

## 🏥 Clinical Applications

### Depression Detection
**Signs to look for:**
- High sadness (>60%)
- Monotone/flat speech
- Low energy
- Neutral emotion (emotionless)

**Action:**
- Flag for review if sadness > 60%
- Recommend depression screening
- Monitor over time

### Anxiety Detection
**Signs to look for:**
- High fear/fearful emotion (>50%)
- Tremor in voice (instability)
- Rapid speech
- High arousal levels

**Action:**
- Flag for review
- Assess anxiety triggers
- Monitor symptoms

### Social Engagement
**Signs to look for:**
- Emotional variety (multiple emotions detected)
- Emotional expressiveness
- Balanced positive/negative

**Action:**
- Encourage social activities
- Monitor isolation patterns

### Cognitive Health
**Signs to look for:**
- Clear speech (high audio quality)
- Coherent thoughts
- Natural rhythm

**Action:**
- Monitor speech clarity over time
- Cognitive screening if declining

---

## 📊 Performance Metrics

### Model Accuracy
- **Accuracy**: 79.57% (on test data)
- **F1 Score**: 79.43%
- **Processing Speed**: 0.08 seconds per second of audio
- **Memory Usage**: ~350 MB

### Audio Requirements
- **Sample Rate**: 16,000 Hz (industry standard)
- **Duration**: 1 second minimum, 20 minutes maximum
- **Chunk Size**: 30 seconds (for long audio)
- **Overlap**: 5 seconds between chunks
- **Formats**: WAV, MP3, M4A, FLAC, OGG, WebM
- **File Size Limit**: 100 MB

### Review Triggers
- **Target Review Rate**: 15-20% of all sessions
- **Low Confidence**: < 40%
- **Poor Audio Quality**: < 40%
- **High Negative Content**: > 70%
- **High Priority Emotions**: > 60% (sad/fearful/angry)

---

## 💾 Data Storage

### Session Files (JSON Format)
Each analysis creates a JSON file containing:
- Session ID and user ID
- Audio file path
- Detected emotion + confidence
- All 8 emotion probabilities
- Audio quality score
- Clinical insights
- Agentic analysis (if available)
- Timestamp

**Example:**
```json
{
  "session_id": "audio_session_12345",
  "user_id": "user_789",
  "dominant_emotion": "happy",
  "dominant_confidence": 0.82,
  "audio_quality_score": 0.75,
  "emotion_distribution": {
    "angry": 0.03,
    "calm": 0.15,
    "disgust": 0.02,
    "fearful": 0.05,
    "happy": 0.82,
    "neutral": 0.08,
    "sad": 0.02,
    "surprised": 0.03
  },
  "clinical_insights": {
    "emotional_valence": 0.8,
    "emotional_arousal": 0.6,
    "clinical_indicators": [...]
  }
}
```

### Database Storage
- Sessions stored in database for quick retrieval
- Audit logs for compliance
- Review requests tracked separately

---

## 🚀 API Endpoints

### Core Analysis
- `POST /audio-analysis/start-session` - Start analysis session
- `POST /audio-analysis/analyze-file` - Upload and analyze audio
- `POST /audio-analysis/analyze-stream` - Stream audio in real-time
- `GET /audio-analysis/session/{id}` - Get session results

### Management
- `GET /audio-analysis/performance` - System statistics
- `GET /audio-analysis/models` - List models
- `POST /audio-analysis/training/weekly` - Trigger model training
- `GET /audio-analysis/supported-formats` - Audio format info

---

## 🔒 Privacy & Security

### Data Protection
- ✓ Encrypted storage
- ✓ Consent-based processing
- ✓ Secure model caching
- ✓ Audit logging (who accessed what, when)
- ✓ HIPAA-compliant handling

### Consent Management
- **With Consent**: Audio metadata saved for improvement
- **Without Consent**: Only aggregated stats kept
- **Default Retention**: 30 days
- **Deletion**: Automatic cleanup of old sessions

---

## 🎯 For Different Audiences

### For Healthcare Providers
- Focus on: Clinical insights, risk indicators, review flags
- Use: Results to inform patient care decisions
- Benefit: Objective emotion tracking over time

### For Patients
- Focus on: Overall emotion, confidence level, recommendations
- Avoid: Technical details (quality metrics, probabilities)
- Benefit: Simple emotion tracking

### For Data Scientists
- Focus on: Model accuracy, training data, optimization
- Use: For improving model performance
- Benefit: Understanding system limitations

### For IT/DevOps
- Focus on: Performance stats, resource usage, error logs
- Use: System monitoring and optimization
- Benefit: Ensuring system health

---

## ⚙️ Configuration (Default Settings)

```python
SPEECH_ANALYSIS_CONFIG = {
    "model_name": "Dpngtm/wav2vec2-emotion-recognition",
    "sample_rate": 16000,              # Hz (samples per second)
    "chunk_duration": 30,              # seconds
    "overlap_duration": 5,             # seconds
    "min_audio_length": 1.0,           # seconds
    "max_audio_length": 1200.0,        # seconds (20 minutes)
    "confidence_threshold": 0.4,       # 40% minimum confidence
    "audio_quality_threshold": 0.6,    # 60% minimum quality
    "review_trigger_threshold": 0.65   # When to flag for review
}
```

---

## 🔄 Processing Workflow

1. **User uploads audio** → Validation
2. **Preprocessing** → Resampling, normalization
3. **Chunking** → Split into 30-second pieces (if needed)
4. **Emotion Detection** → Run through Wav2Vec2
5. **Quality Assessment** → Check audio quality metrics
6. **Result Aggregation** → Combine chunk results
7. **Clinical Insights** → Generate health indicators
8. **AI Analysis** → GPT-4 deep analysis (optional)
9. **Review Decision** → Should expert review?
10. **Save & Return** → Store results, send to user

**Total Time**: ~10-30 seconds for typical audio

---

## 📈 Continuous Improvement

### How the Model Gets Better
1. **Clinical Expert Reviews** Results
2. **Expert Provides Feedback** (correct emotion if wrong)
3. **Weekly Collection** All feedback from past week
4. **Model Retraining** Using correct examples
5. **Validation** Check if accuracy improved
6. **Activation** Use new model if better
7. **Performance Tracking** Monitor improvements over time

### Training Schedule
- **Frequency**: Weekly (if 50+ feedback items)
- **Validation**: 15% of data held for testing
- **Early Stopping**: Stop if no improvement for 10 epochs

---

## ✨ Key Features Summary

✓ **Long Audio Support** - Handles up to 20 minutes (most systems only do 30 seconds)
✓ **Smart Chunking** - Overlapping segments preserve emotional context
✓ **Quality Assurance** - Built-in audio quality checks
✓ **Clinical Grade** - Designed for healthcare use
✓ **Adaptive Model** - Improves from expert feedback
✓ **Multi-Format** - Accepts most common audio formats
✓ **Secure** - Encryption and audit logging included
✓ **Scalable** - Handles many concurrent users
✓ **Expert Review** - Automatic flagging for human review

---

## 🎓 Technical References

- **Model**: facebook/wav2vec2-base-960h (fine-tuned)
- **Fine-Tuned By**: Dpngtm on HuggingFace
- **Training Data**: RAVDESS, CREMA-D, TESS, SAVEE (12,162 samples)
- **Framework**: PyTorch with Transformers
- **Optimization**: Mixed precision (FP16) training

### Papers
- Baevski et al. (2020) - "wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations"
- HuggingFace Transformers Documentation

---

**Last Updated**: 2025
**Version**: 2.0 (Simplified & Enhanced)
**Status**: Production Ready ✅

